Get API KEY from [Cerebras Website]("https://www.cerebras.ai/"). Save to environment as "CEREBRAS_API_KEY"

In [1]:
SYSTEM_PROMPT_STRING = """
You are an NYT Connections solver.
You will be given 16 words.
OUTPUT ONLY the final groups of words and their categories.
Do NOT provide reasoning or explanations under any circumstances.
DO NOT output any text other than the 4 groups and their categories.
Format exactly like this:

GROUP 1: word1, word2, word3, word4
CATEGORY: category_name

GROUP 2: word1, word2, word3, word4
CATEGORY: category_name

GROUP 3: word1, word2, word3, word4
CATEGORY: category_name

GROUP 4: word1, word2, word3, word4
CATEGORY: category_name
"""

USER_PROMPT = "Here are the 16 words: APPLE, PEAR, TABLE, CHAIR, SOFA, BANANA, DESK, PLUM, COUCH, LAMP, ORANGE, STOOL, BED, GRAPE, SHELF, DRAWER"


In [23]:
import os
from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

try:
    response = client.chat.completions.create(
        model="llama3.1-8b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_STRING},
            {"role": "user", "content": USER_PROMPT}
        ],
        temperature=0.1,
        max_tokens=600,
    )

    msg = response.choices[0].message
    output = msg.content or ""
    print(output.strip())

except Exception as e:
    print("ERROR:", e)

GROUP 1: APPLE, PEAR, PLUM, GRAPE
CATEGORY: Fruits

GROUP 2: TABLE, DESK, SHELF, DRAWER
CATEGORY: Furniture

GROUP 3: CHAIR, SOFA, COUCH, BED
CATEGORY: Furniture

GROUP 4: BANANA, ORANGE, LAMP, STOOL
CATEGORY: Objects


In [21]:
import re

def call_llm(system_prompt, user_prompt,
                        model = "llama3.1-8b",
                        temperature = 0.1,
                        max_tokens = 600):
    """
    Sends a chat request to the Cerebras API and returns the response content.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        msg = response.choices[0].message
        return (msg.content or "").strip()
    
    except Exception as e:
        return f"ERROR: {e}"

def convert_puzzle_to_prompt(words16):
    word_list_str = ", ".join(words16)
    return f"Here are the 16 words: {word_list_str}"

def parse_response_to_pred_groups(response):
    pattern = r"GROUP \d+: (.+)"
    
    groups = re.findall(pattern, response)
    parsed_groups = [ [word.strip() for word in group.split(",")] for group in groups ]
    
    return parsed_groups

In [22]:
from conn import load_connections_from_hf

hf_split = load_connections_from_hf()
print("puzzles:", len(hf_split))

puzzles: 652


In [25]:
def solve_puzzle(words16, model="llama3.1-8b", temperature=0.1, max_tokens=600):
    user_prompt = convert_puzzle_to_prompt(words16)
    response = call_llm(SYSTEM_PROMPT_STRING, user_prompt, model=model, 
                        temperature=temperature, max_tokens=max_tokens)
    return parse_response_to_pred_groups(response)

row0 = hf_split[0]
words16 = row0["words"]
pred_groups = solve_puzzle(words16)

print("\nPredicted groups:")
for g in pred_groups:
    print(g)

print("\nGold groups:")
for ans in row0["answers"]:
    print(ans["answerDescription"], "->", ans["words"])


Predicted groups:
['LASER', 'COIL', 'SOLAR PANEL', 'WIND']
['PLUCK', 'WAX', 'WRAP', 'THREAD']
['HONEYCOMB', 'SPREADSHEET', 'SCHOOL', 'ORGANISM']
['BALL', 'MOVIE', 'VITAMIN', 'SPOOL']

Gold groups:
REMOVE, AS BODY HAIR -> ['LASER', 'PLUCK', 'THREAD', 'WAX']
TWIST AROUND -> ['COIL', 'SPOOL', 'WIND', 'WRAP']
THINGS MADE OF CELLS -> ['HONEYCOMB', 'ORGANISM', 'SOLAR PANEL', 'SPREADSHEET']
B-___ -> ['BALL', 'MOVIE', 'SCHOOL', 'VITAMIN']


In [26]:
from conn import (
    accuracy_min_swaps,
    accuracy_zero_one,
    evaluate,
    gold_groups_from_row,
)

N_EVAL = 10
acc, n_eval = evaluate(
    hf_split,
    metric_fn=accuracy_zero_one,
    solver_fn=solve_puzzle,
    max_samples=N_EVAL,
    gold_from_row=gold_groups_from_row,
)
mean_swaps, _ = evaluate(
    hf_split,
    metric_fn=accuracy_min_swaps,
    solver_fn=solve_puzzle,
    max_samples=N_EVAL,
    gold_from_row=gold_groups_from_row,
)
print(f"Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
print(f"Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")

Zero-one accuracy: 0.4000  (n=10, requested=10)
Mean 1-1 swaps to correct: 1.80  (n=10)
